# NASDAQ Multivariate Data Pipeline

**Sources**
- `yfinance` — NASDAQ OHLCV, VIX, 10Y Treasury Yield, DXY
- `FRED API` — Fed Funds Rate, Unemployment Rate, CPI
- `pandas-ta` — Technical indicators (RSI, MACD, Bollinger Bands, etc.)

**Output:** `data/processed/nasdaq_multivariate.csv`

# Install dependencies if needed
# !pip install yfinance pandas-ta fredapi

## 1. Imports & Config

In [24]:
import os
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_ta as ta
from fredapi import Fred

# ── Config ────────────────────────────────────────────────────────────────────
FRED_API_KEY = "c957715a41210fa05f882d9ea4763c10"   #
START_DATE   = "1989-01-01"
END_DATE     = "2026-03-20"

os.makedirs("data/raw",       exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("Config ready.")

Config ready.


## 2. Market Data (yfinance)

Daily data for NASDAQ Composite + three market signals:
- `^VIX` — CBOE Volatility Index
- `^TNX` — 10-Year Treasury Yield  
- `DX-Y.NYB` — US Dollar Index (DXY)

In [25]:
# ── NASDAQ OHLCV ──────────────────────────────────────────────────────────────
nasdaq = yf.download("^IXIC", start=START_DATE, end=END_DATE,
                     auto_adjust=True, progress=False)
nasdaq.columns = nasdaq.columns.droplevel(1)
nasdaq.to_csv("data/raw/nasdaq.csv")
print(f"NASDAQ: {nasdaq.shape}  |  {nasdaq.index[0].date()} → {nasdaq.index[-1].date()}")
nasdaq.tail(3)

/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


NASDAQ: (9372, 5)  |  1989-01-03 → 2026-03-19


Price,Close,High,Low,Open,Volume
Date,,,,,
2026-03-17,22479.529297,22569.640625,22409.070312,22458.029297,8505210000
2026-03-18,22152.419922,22461.759766,22144.759766,22421.960938,9325170000
2026-03-19,22090.689453,22187.060547,21851.050781,21871.039062,8547870000


In [26]:
# ── Market signals ────────────────────────────────────────────────────────────
signal_tickers = {
    "^VIX"    : "VIX",
    "^TNX"    : "TNX",
    "DX-Y.NYB": "DXY",
}

signals = {}
for ticker, name in signal_tickers.items():
    df = yf.download(ticker, start=START_DATE, end=END_DATE,
                     auto_adjust=True, progress=False)
    df.columns = df.columns.droplevel(1)
    signals[name] = df["Close"]
    df.to_csv(f"data/raw/{name.lower()}.csv")
    print(f"{name}: {df.shape[0]} rows, {df['Close'].isna().sum()} NaNs")

signals_df = pd.DataFrame(signals)

# Join to NASDAQ and forward-fill any gaps (e.g. holidays)
market_df = nasdaq.join(signals_df, how="left")
market_df[["VIX", "TNX", "DXY"]] = market_df[["VIX", "TNX", "DXY"]].ffill()

print(f"\nMarket DataFrame: {market_df.shape}")
market_df.tail(3)

/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


VIX: 9120 rows, 0 NaNs


/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


TNX: 9339 rows, 0 NaNs


/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


DXY: 9489 rows, 0 NaNs

Market DataFrame: (9372, 8)


,Close,High,Low,Open,Volume,VIX,TNX,DXY
Date,,,,,,,,
2026-03-17,22479.529297,22569.640625,22409.070312,22458.029297,8505210000,22.370001,4.202,99.580002
2026-03-18,22152.419922,22461.759766,22144.759766,22421.960938,9325170000,25.090000,4.259,100.089996
2026-03-19,22090.689453,22187.060547,21851.050781,21871.039062,8547870000,24.059999,4.281,99.230003


## 3. Macro Data (FRED)

Monthly series forward-filled to daily trading calendar.  
Forward-filling is the standard way to avoid look-ahead bias — each day
sees only the most recently *published* value.

In [27]:
fred = Fred(api_key=FRED_API_KEY)

series_map = {
    "FEDFUNDS" : "FedRate",      # Federal Funds Rate (monthly)
    "UNRATE"   : "Unemployment", # Unemployment Rate (monthly)
    "CPIAUCSL" : "CPI",          # Consumer Price Index (monthly)
}

macro_dict = {}
for fred_id, col_name in series_map.items():
    s = fred.get_series(fred_id,
                        observation_start=START_DATE,
                        observation_end=END_DATE)
    s.name = col_name
    macro_dict[col_name] = s
    s.to_csv(f"data/raw/{col_name.lower()}.csv", header=True)
    print(f"{col_name}: {len(s)} monthly observations")

macro_monthly = pd.DataFrame(macro_dict)

# Reindex to daily trading calendar, then forward-fill
macro_daily = (
    macro_monthly
    .reindex(market_df.index, method=None)
    .ffill()
)

# Add month-over-month CPI change as inflation proxy
macro_daily["CPI_MoM"] = macro_daily["CPI"].pct_change()

print(f"\nMacro DataFrame (daily): {macro_daily.shape}")
macro_daily.tail(3)

FedRate: 447 monthly observations
Unemployment: 447 monthly observations
CPI: 447 monthly observations

Macro DataFrame (daily): (9372, 4)


,FedRate,Unemployment,CPI,CPI_MoM
Date,,,,
2026-03-17,3.72,4.4,326.031,0.0
2026-03-18,3.72,4.4,326.031,0.0
2026-03-19,3.72,4.4,326.031,0.0


## 4. Join Market + Macro

In [28]:
df = market_df.join(macro_daily, how="left")
print(f"Combined DataFrame: {df.shape}")
df.tail(3)

Combined DataFrame: (9372, 12)


,Close,High,Low,Open,Volume,VIX,TNX,DXY,FedRate,Unemployment,CPI,CPI_MoM
Date,,,,,,,,,,,,
2026-03-17,22479.529297,22569.640625,22409.070312,22458.029297,8505210000,22.370001,4.202,99.580002,3.72,4.4,326.031,0.0
2026-03-18,22152.419922,22461.759766,22144.759766,22421.960938,9325170000,25.090000,4.259,100.089996,3.72,4.4,326.031,0.0
2026-03-19,22090.689453,22187.060547,21851.050781,21871.039062,8547870000,24.059999,4.281,99.230003,3.72,4.4,326.031,0.0


## 5. Technical Indicators

Derived from NASDAQ OHLCV using `pandas-ta`.

| Group | Features |
|-------|----------|
| Returns | Log return, lagged returns (1/2/5 days) |
| Moving Averages | SMA 20/50, EMA 12/26 |
| Momentum | RSI-14 |
| Trend | MACD, MACD Signal, MACD Histogram |
| Volatility | Bollinger Bands (width, position) |
| Volume | Daily % change, ratio to 20-day avg |

In [29]:
close  = df["Close"]
high   = df["High"]
low    = df["Low"]
volume = df["Volume"]

# ── Returns ───────────────────────────────────────────────────────────────────
df["Return"]        = np.log(close / close.shift(1))
df["Return_lag1"]   = df["Return"].shift(1)
df["Return_lag2"]   = df["Return"].shift(2)
df["Return_lag5"]   = df["Return"].shift(5)

# ── Moving averages ───────────────────────────────────────────────────────────
df["SMA_20"]        = ta.sma(close, length=20)
df["SMA_50"]        = ta.sma(close, length=50)
df["EMA_12"]        = ta.ema(close, length=12)
df["EMA_26"]        = ta.ema(close, length=26)

# Price position relative to MAs
df["Price_SMA20"]   = close / df["SMA_20"] - 1
df["Price_SMA50"]   = close / df["SMA_50"] - 1

# ── RSI ───────────────────────────────────────────────────────────────────────
df["RSI_14"]        = ta.rsi(close, length=14)

# ── MACD ──────────────────────────────────────────────────────────────────────
macd = ta.macd(close, fast=12, slow=26, signal=9)
df["MACD"]          = macd["MACD_12_26_9"]
df["MACD_Signal"]   = macd["MACDs_12_26_9"]
df["MACD_Hist"]     = macd["MACDh_12_26_9"]

# ── Bollinger Bands ───────────────────────────────────────────────────────────
bb = ta.bbands(close, length=20, std=2)
df["BB_upper"]      = bb["BBU_20_2.0_2.0"]
df["BB_mid"]        = bb["BBM_20_2.0_2.0"]
df["BB_lower"]      = bb["BBL_20_2.0_2.0"]
df["BB_width"]      = (df["BB_upper"] - df["BB_lower"]) / df["BB_mid"]
df["BB_pos"]        = (close - df["BB_lower"]) / (df["BB_upper"] - df["BB_lower"])

# ── Volume ────────────────────────────────────────────────────────────────────
df["Volume_change"] = volume.pct_change()
df["Volume_SMA20"]  = ta.sma(volume.astype(float), length=20)
df["Volume_ratio"]  = volume / df["Volume_SMA20"]

print(f"Shape after indicators: {df.shape}")
df[["Return", "RSI_14", "MACD", "BB_width", "Volume_ratio"]].tail(3)

Shape after indicators: (9372, 34)


,Return,RSI_14,MACD,BB_width,Volume_ratio
Date,,,,,
2026-03-17,0.004697,45.393148,-176.925492,0.041967,0.968617
2026-03-18,-0.014658,40.306694,-194.003179,0.046216,1.052294
2026-03-19,-0.002791,39.409236,-210.096639,0.050775,0.956148


## 6. Labels

| Column | Type | Description |
|--------|------|-------------|
| `NextReturn` | float | Next-day log return (regression target) |
| `NextDirection` | 0/1 | 1 if next-day return > 0 (classification target) |

In [30]:
df["NextReturn"]    = df["Return"].shift(-1)
df["NextDirection"] = (df["NextReturn"] > 0).astype(int)

print("Label distribution:")
print(df["NextDirection"].value_counts())
print(f"\nClass balance: {df['NextDirection'].mean():.3f} (fraction up days)")

Label distribution:
NextDirection
1    5175
0    4197
Name: count, dtype: int64

Class balance: 0.552 (fraction up days)


## 7. Drop Warm-up Rows & Save

In [31]:
# Drop rows where long-window indicators are still NaN
# SMA_50 has the longest warm-up (50 days)
df.dropna(subset=["SMA_50", "RSI_14", "MACD", "FedRate", "NextReturn", "NextDirection"], inplace=True)

out_path = "data/processed/nasdaq_multivariate.csv"
df.to_csv(out_path)

print(f"Saved → {out_path}")
print(f"Shape : {df.shape}")
print(f"Period: {df.index[0].date()} → {df.index[-1].date()}")
print(f"\nAll columns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

Saved → data/processed/nasdaq_multivariate.csv
Shape : (9322, 36)
Period: 1989-03-14 → 2026-03-18

All columns (36):
   1. Close
   2. High
   3. Low
   4. Open
   5. Volume
   6. VIX
   7. TNX
   8. DXY
   9. FedRate
  10. Unemployment
  11. CPI
  12. CPI_MoM
  13. Return
  14. Return_lag1
  15. Return_lag2
  16. Return_lag5
  17. SMA_20
  18. SMA_50
  19. EMA_12
  20. EMA_26
  21. Price_SMA20
  22. Price_SMA50
  23. RSI_14
  24. MACD
  25. MACD_Signal
  26. MACD_Hist
  27. BB_upper
  28. BB_mid
  29. BB_lower
  30. BB_width
  31. BB_pos
  32. Volume_change
  33. Volume_SMA20
  34. Volume_ratio
  35. NextReturn
  36. NextDirection


## 8. Sanity Check

In [32]:
# Missing value summary
missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("No missing values.")
else:
    print("Columns with missing values:")
    print(missing)

print("\nBasic stats:")
df[df.columns].describe().round(4)

Columns with missing values:
VIX    203
dtype: int64

Basic stats:


,Close,High,Low,Open,Volume,VIX,TNX,DXY,FedRate,Unemployment,...,BB_upper,BB_mid,BB_lower,BB_width,BB_pos,Volume_change,Volume_SMA20,Volume_ratio,NextReturn,NextDirection
count,9322.0000,9322.0000,9322.0000,9322.0000,9.322000e+03,9119.0000,9322.0000,9322.0000,9322.0000,9322.0000,...,9322.0000,9322.0000,9322.0000,9322.0000,9322.0000,9322.0000,9.322000e+03,9322.0000,9322.0000,9322.0000
mean,4668.7018,4700.8381,4632.2944,4668.6806,2.134272e+09,19.4550,4.3329,92.6659,3.0394,5.6582,...,4844.7411,4646.1690,4447.5968,0.0868,0.5784,0.0182,2.125235e+09,1.0060,0.0004,0.5515
std,5164.9165,5198.4909,5126.2857,5164.6993,2.176236e+09,7.7678,2.0065,10.0064,2.5031,1.7231,...,5338.9627,5132.7383,4930.9993,0.0548,0.3202,0.2283,1.941357e+09,0.1737,0.0144,0.4974
min,325.4000,327.9000,323.0000,325.1000,4.421000e+07,9.1400,0.4990,71.3300,0.0500,3.5000,...,344.8226,333.7150,319.2491,0.0098,-0.3733,-0.9139,1.040960e+08,0.1221,-0.1315,0.0000
25%,1457.7525,1474.8549,1435.3050,1453.8975,7.575825e+08,13.9550,2.6400,84.7800,0.3300,4.4000,...,1559.8483,1445.5521,1340.9791,0.0505,0.3246,-0.0813,7.734135e+08,0.9096,-0.0056,0.0000
50%,2429.6300,2449.9401,2408.1200,2429.3049,1.797970e+09,17.6100,4.1960,93.0450,2.9600,5.4000,...,2525.0696,2432.7253,2314.9262,0.0720,0.6484,0.0013,1.838944e+09,0.9974,0.0011,1.0000
75%,5391.9073,5402.6249,5370.3176,5390.6351,2.277482e+09,22.7350,5.8047,99.0600,5.2500,6.5000,...,5467.0392,5312.0311,5205.8942,0.1060,0.8442,0.0918,2.198698e+09,1.0884,0.0074,1.0000
max,23958.4707,24019.9902,23775.4902,23987.2891,9.397454e+10,82.6900,9.5300,120.9000,9.8500,14.8000,...,24301.7580,23538.5463,23132.9778,0.4676,1.2542,9.7641,1.401300e+10,6.8462,0.1325,1.0000


In [33]:
print(df.columns)
df.tail(3)

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'VIX', 'TNX', 'DXY',
       'FedRate', 'Unemployment', 'CPI', 'CPI_MoM', 'Return', 'Return_lag1',
       'Return_lag2', 'Return_lag5', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26',
       'Price_SMA20', 'Price_SMA50', 'RSI_14', 'MACD', 'MACD_Signal',
       'MACD_Hist', 'BB_upper', 'BB_mid', 'BB_lower', 'BB_width', 'BB_pos',
       'Volume_change', 'Volume_SMA20', 'Volume_ratio', 'NextReturn',
       'NextDirection'],
      dtype='str')


,Close,High,Low,Open,Volume,VIX,TNX,DXY,FedRate,Unemployment,...,BB_upper,BB_mid,BB_lower,BB_width,BB_pos,Volume_change,Volume_SMA20,Volume_ratio,NextReturn,NextDirection
Date,,,,,,,,,,,,,,,,,,,,,
2026-03-16,22374.179688,22521.589844,22316.630859,22340.390625,8208800000,23.510000,4.220,99.709999,3.72,4.4,...,23129.762334,22660.041602,22190.320869,0.041458,0.195711,-0.008448,8.738228e+09,0.939412,0.004697,1
2026-03-17,22479.529297,22569.640625,22409.070312,22458.029297,8505210000,22.370001,4.202,99.580002,3.72,4.4,...,23130.483871,22655.099023,22179.714176,0.041967,0.315339,0.036109,8.780774e+09,0.968617,-0.014658,0
2026-03-18,22152.419922,22461.759766,22144.759766,22421.960938,9325170000,25.090000,4.259,100.089996,3.72,4.4,...,23147.856972,22625.038477,22102.219981,0.046216,0.048009,0.096407,8.861752e+09,1.052294,-0.002791,0


In [34]:
feature_cols = [
    # price/momentum
    "Return", "Return_lag1", "Return_lag2", "Return_lag5", "Price_SMA20",
    # technical indicators + volume
    "RSI_14", "MACD_Hist", "BB_pos", "BB_width", "Price_SMA50",
    "Volume_ratio", "Volume_change",
    # market emotion
    "VIX", "TNX",
    # macros
    "FedRate", "CPI_MoM",
    # label
    "NextReturn", "NextDirection",
]